In [15]:
!pip install -q trimesh python-fcl smolagents gradio

import json, trimesh
from google.colab import userdata

def bloc(nom, dims, pos):
    m = trimesh.creation.box(extents=dims); m.apply_translation(pos); return m

def cylindre(r, h, pos):
    m = trimesh.creation.cylinder(radius=r, height=h); m.apply_translation(pos); return m

pieces = {
    "socle":     bloc("socle", [100, 100, 10], [0, 0, 0]),      # z de -5 à 5
    "support_a": bloc("a", [20, 20, 20], [-30, 0, 16.2]),       # jeu de 1,2 mm → non conforme
    "support_b": bloc("b", [20, 20, 20], [30, 0, 19.8]),        # jeu de 4,8 mm → conforme
    "vis_1":     cylindre(3, 30, [0, 30, 10]),                  # traverse le socle → interférence
}

def exporter(rouges=()):
    scene = trimesh.Scene()
    for nom, m in pieces.items():
        m = m.copy()
        m.visual.face_colors = [220, 40, 40, 255] if nom in rouges else [170, 170, 175, 255]
        scene.add_geometry(m, geom_name=nom)
    scene.export("assemblage.glb")
    return "assemblage.glb"

exporter()

'assemblage.glb'

In [16]:
from smolagents import tool

a_surligner = set()

@tool
def lister_pieces() -> str:
    """Liste les pièces de l'assemblage avec leurs dimensions en mm."""
    return json.dumps({n: [round(x, 1) for x in m.extents] for n, m in pieces.items()})

@tool
def distance_min(piece_a: str, piece_b: str) -> float:
    """Calcule la distance minimale en mm entre deux pièces. Renvoie 0 si elles se touchent ou se chevauchent.

    Args:
        piece_a: nom de la première pièce
        piece_b: nom de la deuxième pièce
    """
    cm = trimesh.collision.CollisionManager()
    cm.add_object(piece_a, pieces[piece_a])
    return round(cm.min_distance_single(pieces[piece_b]), 2)

@tool
def detecter_interferences() -> str:
    """Renvoie la liste des paires de pièces qui se chevauchent (interférences)."""
    cm = trimesh.collision.CollisionManager()
    for n, m in pieces.items():
        cm.add_object(n, m)
    _, paires = cm.in_collision_internal(return_names=True)
    return json.dumps(sorted(list(p) for p in paires))

@tool
def surligner_pieces(noms: list) -> str:
    """Marque des pièces en rouge dans la vue 3D, par exemple les pièces non conformes.

    Args:
        noms: liste des noms de pièces à surligner
    """
    a_surligner.update(noms)
    return f"Pièces surlignées : {noms}"

In [17]:
from smolagents import CodeAgent, OpenAIServerModel

model = OpenAIServerModel(model_id="openai/gpt-oss-120b",
                          api_base="https://api.groq.com/openai/v1",
                          api_key=userdata.get("GROQ_API_KEY"))

agent = CodeAgent(tools=[lister_pieces, distance_min, detecter_interferences, surligner_pieces],
                  model=model, max_steps=6)

CONSIGNES = ("Tu es un assistant de revue de conception. Utilise uniquement les résultats des outils. "
             "Ne propose aucune solution technique. Surligne les pièces non conformes. "
             "Termine avec final_answer() en donnant un rapport court : mesures, conformité, interférences.")

In [18]:
import gradio as gr

def revue(demande):
    a_surligner.clear()
    rapport = agent.run(CONSIGNES + "\n\nDemande : " + demande)
    return exporter(a_surligner), str(rapport)

with gr.Blocks(title="3D Design Review Agent") as demo:
    gr.Markdown("## 3D Design Review Agent")
    vue = gr.Model3D(value="assemblage.glb", label="Maquette")
    demande = gr.Textbox(value="Vérifie que toutes les pièces respectent un jeu minimal de 2 mm avec le socle, "
                               "et détecte les interférences.", label="Demande")
    bouton = gr.Button("Lancer la revue")
    rapport = gr.Markdown()
    bouton.click(revue, inputs=demande, outputs=[vue, rapport])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


2026/10/28 10:40:47 [W] [service.go:132] login to server failed: dial tcp 44.237.78.176:7000: connect: connection refused


<IPython.core.display.Javascript object>